# 03 — Regression Modeling

This notebook compares a porosity-only baseline, Lasso regression, random forest regression, and an optional geometry-inclusive model using grouped validation to reduce fabrication-batch leakage.

## 1. Imports and Modeling Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

print("✓ Modeling imports successful")
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")

✓ Modeling imports successful
NumPy version: 2.4.6
pandas version: 3.0.3


## 2. Load Modeling Tables

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

MAIN_FEATURE_PATH = DATA_DIR / "main_features.csv"
GEOMETRY_FEATURE_PATH = DATA_DIR / "geometry_features.csv"
TARGET_GROUP_PATH = DATA_DIR / "target_and_groups.csv"

main_features = pd.read_csv(MAIN_FEATURE_PATH)
geometry_features = pd.read_csv(GEOMETRY_FEATURE_PATH)
target_groups = pd.read_csv(TARGET_GROUP_PATH)

target = target_groups["Maximum Stress (MPa)"]
groups = target_groups["Group ID"]

print("✓ Modeling tables loaded successfully")
print(f"Main features shape: {main_features.shape}")
print(f"Geometry features shape: {geometry_features.shape}")
print(f"Target shape: {target.shape}")
print(f"Number of fabrication groups: {groups.nunique()}")

✓ Modeling tables loaded successfully
Main features shape: (256, 10)
Geometry features shape: (256, 11)
Target shape: (256,)
Number of fabrication groups: 44


## 3. Grouped Cross-Validation Strategy

In [3]:
group_cv = GroupKFold(n_splits=5)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

print("✓ Grouped cross-validation configured")
print("Number of folds: 5")
print("Grouping variable: Group ID")

✓ Grouped cross-validation configured
Number of folds: 5
Grouping variable: Group ID


## 4. Porosity-Only Linear Baseline

In [4]:
porosity_features = main_features[["Porosity (%)"]].copy()

porosity_baseline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression())
    ]
)

porosity_cv_results = cross_validate(
    estimator=porosity_baseline,
    X=porosity_features,
    y=target,
    groups=groups,
    cv=group_cv,
    scoring=scoring,
    return_train_score=False
)

porosity_baseline_summary = {
    "Model": "Porosity-only Linear Regression",
    "MAE (MPa)": -porosity_cv_results["test_mae"].mean(),
    "RMSE (MPa)": -porosity_cv_results["test_rmse"].mean(),
    "R²": porosity_cv_results["test_r2"].mean()
}

print("✓ Porosity-only baseline evaluated")
for metric, value in porosity_baseline_summary.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.3f}")

✓ Porosity-only baseline evaluated
Model: Porosity-only Linear Regression
MAE (MPa): 15.181
RMSE (MPa): 17.278
R²: -0.129


## 5. Preprocessing for Multivariable Models

In [5]:
categorical_features = main_features.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = main_features.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("✓ Preprocessing pipeline created")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

✓ Preprocessing pipeline created
Numeric features: 7
Categorical features: 3


## 6. Lasso Regression

In [6]:
lasso_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Lasso(alpha=0.1, max_iter=10000))
    ]
)

lasso_cv_results = cross_validate(
    estimator=lasso_model,
    X=main_features,
    y=target,
    groups=groups,
    cv=group_cv,
    scoring=scoring,
    return_train_score=False
)

lasso_summary = {
    "Model": "Lasso Regression",
    "MAE (MPa)": -lasso_cv_results["test_mae"].mean(),
    "RMSE (MPa)": -lasso_cv_results["test_rmse"].mean(),
    "R²": lasso_cv_results["test_r2"].mean()
}

print("✓ Lasso regression evaluated")
for metric, value in lasso_summary.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.3f}")

✓ Lasso regression evaluated
Model: Lasso Regression
MAE (MPa): 14.888
RMSE (MPa): 18.045
R²: -0.521


## 7. Random Forest Regression

In [7]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=2,
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

random_forest_cv_results = cross_validate(
    estimator=random_forest_model,
    X=main_features,
    y=target,
    groups=groups,
    cv=group_cv,
    scoring=scoring,
    return_train_score=False
)

random_forest_summary = {
    "Model": "Random Forest Regression",
    "MAE (MPa)": -random_forest_cv_results["test_mae"].mean(),
    "RMSE (MPa)": -random_forest_cv_results["test_rmse"].mean(),
    "R²": random_forest_cv_results["test_r2"].mean()
}

print("✓ Random forest regression evaluated")
for metric, value in random_forest_summary.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.3f}")

✓ Random forest regression evaluated
Model: Random Forest Regression
MAE (MPa): 11.976
RMSE (MPa): 16.288
R²: -0.085


## 8. Geometry-Inclusive Random Forest

In [8]:
geometry_categorical_features = geometry_features.select_dtypes(
    include=["object", "string"]
).columns.tolist()

geometry_numeric_features = geometry_features.select_dtypes(
    include=["number"]
).columns.tolist()

geometry_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]
            ),
            geometry_numeric_features
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            geometry_categorical_features
        )
    ]
)

geometry_random_forest = Pipeline(
    steps=[
        ("preprocessor", geometry_preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=2,
                random_state=SEED,
                n_jobs=-1
            )
        )
    ]
)

geometry_cv_results = cross_validate(
    estimator=geometry_random_forest,
    X=geometry_features,
    y=target,
    groups=groups,
    cv=group_cv,
    scoring=scoring,
    return_train_score=False
)

geometry_summary = {
    "Model": "Geometry-Inclusive Random Forest",
    "MAE (MPa)": -geometry_cv_results["test_mae"].mean(),
    "RMSE (MPa)": -geometry_cv_results["test_rmse"].mean(),
    "R²": geometry_cv_results["test_r2"].mean()
}

print("✓ Geometry-inclusive random forest evaluated")
for metric, value in geometry_summary.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.3f}")

✓ Geometry-inclusive random forest evaluated
Model: Geometry-Inclusive Random Forest
MAE (MPa): 11.004
RMSE (MPa): 15.537
R²: -0.169


## 9. Compare Model Performance

In [9]:
model_comparison = pd.DataFrame([
    porosity_baseline_summary,
    lasso_summary,
    random_forest_summary,
    geometry_summary
])

model_comparison = model_comparison.sort_values(
    by="MAE (MPa)",
    ascending=True
).reset_index(drop=True)

print("✓ Model comparison table created")
display(model_comparison.round(3))

✓ Model comparison table created


,Model,MAE (MPa),RMSE (MPa),R²
0,Geometry-Inclusive Random Forest,11.004,15.537,-0.169
1,Random Forest Regression,11.976,16.288,-0.085
2,Lasso Regression,14.888,18.045,-0.521
3,Porosity-only Linear Regression,15.181,17.278,-0.129


## 10. Examine Fold-to-Fold Performance

In [10]:
fold_performance = pd.DataFrame({
    "Fold": np.arange(1, 6),
    "Porosity Baseline MAE (MPa)": -porosity_cv_results["test_mae"],
    "Lasso MAE (MPa)": -lasso_cv_results["test_mae"],
    "Random Forest MAE (MPa)": -random_forest_cv_results["test_mae"],
    "Geometry RF MAE (MPa)": -geometry_cv_results["test_mae"],
    "Random Forest R²": random_forest_cv_results["test_r2"],
    "Geometry RF R²": geometry_cv_results["test_r2"]
})

print("✓ Fold-level performance table created")
display(fold_performance.round(3))

✓ Fold-level performance table created


,Fold,Porosity Baseline MAE (MPa),Lasso MAE (MPa),Random Forest MAE (MPa),Geometry RF MAE (MPa),Random Forest R²,Geometry RF R²
0,1,15.324,16.388,11.929,10.458,-0.211,-0.120
1,2,18.068,17.291,15.676,15.068,-0.393,-0.258
2,3,9.610,16.046,10.082,12.740,-0.448,-1.371
3,4,16.074,11.241,13.904,12.506,0.021,-0.006
4,5,16.827,13.475,8.289,4.249,0.605,0.909


## 11. Generate Out-of-Fold Predictions

In [11]:
from sklearn.model_selection import cross_val_predict

random_forest_oof_predictions = cross_val_predict(
    estimator=random_forest_model,
    X=main_features,
    y=target,
    groups=groups,
    cv=group_cv,
    n_jobs=-1
)

geometry_oof_predictions = cross_val_predict(
    estimator=geometry_random_forest,
    X=geometry_features,
    y=target,
    groups=groups,
    cv=group_cv,
    n_jobs=-1
)

print("✓ Out-of-fold predictions generated")
print(f"Random forest predictions: {random_forest_oof_predictions.shape}")
print(f"Geometry RF predictions: {geometry_oof_predictions.shape}")

✓ Out-of-fold predictions generated
Random forest predictions: (256,)
Geometry RF predictions: (256,)


## 12. Evaluate Out-of-Fold Predictions

In [12]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE (MPa)": mean_absolute_error(y_true, y_pred),
        "RMSE (MPa)": mean_squared_error(y_true, y_pred) ** 0.5,
        "R²": r2_score(y_true, y_pred)
    }


random_forest_oof_metrics = regression_metrics(
    target,
    random_forest_oof_predictions
)

geometry_oof_metrics = regression_metrics(
    target,
    geometry_oof_predictions
)

oof_comparison = pd.DataFrame([
    {
        "Model": "Random Forest Regression",
        **random_forest_oof_metrics
    },
    {
        "Model": "Geometry-Inclusive Random Forest",
        **geometry_oof_metrics
    }
])

print("✓ Out-of-fold metrics calculated")
display(oof_comparison.round(3))

✓ Out-of-fold metrics calculated


,Model,MAE (MPa),RMSE (MPa),R²
0,Random Forest Regression,11.969,16.736,0.042
1,Geometry-Inclusive Random Forest,10.969,16.403,0.080


## 13. Save Modeling Results

In [13]:
MODEL_RESULTS_PATH = DATA_DIR / "model_comparison.csv"
OOF_PREDICTIONS_PATH = DATA_DIR / "oof_predictions.csv"

model_comparison.to_csv(MODEL_RESULTS_PATH, index=False)

oof_predictions = pd.DataFrame({
    "Observed Maximum Stress (MPa)": target,
    "Random Forest Prediction (MPa)": random_forest_oof_predictions,
    "Geometry RF Prediction (MPa)": geometry_oof_predictions,
    "Group ID": groups
})

oof_predictions.to_csv(OOF_PREDICTIONS_PATH, index=False)

print("✓ Modeling results saved")
print(f"Model comparison: {MODEL_RESULTS_PATH.name}")
print(f"Out-of-fold predictions: {OOF_PREDICTIONS_PATH.name}")

✓ Modeling results saved
Model comparison: model_comparison.csv
Out-of-fold predictions: oof_predictions.csv


## 14. Fit the Final Geometry-Inclusive Random Forest

In [14]:
geometry_random_forest.fit(
    geometry_features,
    target
)

feature_names = geometry_random_forest.named_steps[
    "preprocessor"
].get_feature_names_out()

feature_importances = geometry_random_forest.named_steps[
    "model"
].feature_importances_

feature_importance_table = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": feature_importances
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("✓ Final geometry-inclusive random forest fitted")
display(feature_importance_table.head(15))

✓ Final geometry-inclusive random forest fitted


,Feature,Importance
0,numeric__Cross-Sectional Area (mm^2),0.269981
1,numeric__Volume Loss (%),0.172415
2,numeric__Mass Loss (%),0.161900
3,numeric__Final Mass (g),0.121606
4,numeric__Porosity (%),0.107923
5,numeric__Mass Loss (g),0.091335
6,numeric__Volume Loss (cm^3),0.024897
7,numeric__Final Volume (cm^3),0.017054
8,categorical__Binder Composition_PVA PEG200,0.010107
9,categorical__Binder Composition_PVA,0.008926


## 15. Save Feature-Importance Results

In [15]:
FEATURE_IMPORTANCE_PATH = DATA_DIR / "feature_importance.csv"

feature_importance_table.to_csv(
    FEATURE_IMPORTANCE_PATH,
    index=False
)

print("✓ Feature-importance table saved")
print(f"Saved file: {FEATURE_IMPORTANCE_PATH.name}")

✓ Feature-importance table saved
Saved file: feature_importance.csv


## 16. Notebook Summary

Four regression approaches were evaluated using five-fold GroupKFold cross-validation with fabrication group as the grouping variable.

Key findings:

- The porosity-only linear baseline performed poorly across unseen fabrication groups.
- Lasso regression did not improve grouped generalization.
- Random forest regression reduced MAE relative to the linear models.
- Adding cross-sectional area further reduced out-of-fold MAE to approximately 11 MPa.
- Out-of-fold R2 remained low, showing that prediction across unseen fabrication groups is difficult.
- Model performance varied substantially between folds, indicating strong fabrication-group effects.
- Cross-sectional area, volume loss, mass loss, final mass, and porosity were among the most important predictors in the fitted geometry-inclusive random forest.
- Compressive load was excluded because it directly determines the target stress.

The following files were saved for Notebook 4:

- `data/model_comparison.csv`
- `data/oof_predictions.csv`
- `data/feature_importance.csv`